In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys

from pathlib import Path
from typing import Tuple

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.vis as vis

from config_handlers.eval_handler import EvalHandler
from config_handlers.rk_handler import RKHandler
from config_handlers.sk_handler import SKHandler

## Setup

In [ ]:
eval_hdl = EvalHandler()

model_list = eval_hdl.get_model_list()
eval_hdl.set_curr_model(model_list[0])
ds_list = eval_hdl.get_ds_list()

all_ds = ds_list * len(model_list)
all_models = model_list * len(ds_list)
all_models.sort()

OPEN_DATASETS = ["llava-bench", "mmbench"]
MC_DATASETS = ["seed", "vqav2"]

CHARTS_DIR = Path("behaviour_charts")
data_io.make_dir(CHARTS_DIR)

[Errno 17] File exists: 'behaviour_charts'


In [3]:
def load_triples_data(emb_model="mpnet") -> Tuple[dict, dict]:
    samples = {m: {ds: {} for ds in ds_list} for m in model_list}
    summaries = {m: {ds: {} for ds in ds_list} for m in model_list}
    for model, ds in zip(all_models, all_ds):
        base_fp = Path("raw_stats", f"sim_overlap_{emb_model}")
        samples[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "samples.json")
        )
        summaries[model][ds] = data_io.load_json(
            base_fp.joinpath(model, ds, "summary.json")
        )

    return samples, summaries

Loading eval responses for correlation between triple similarity and performance

In [4]:
def load_eval_responses():
    data = {m: {ds: {}} for m in model_list for ds in ds_list}
    for model, ds in zip(all_models, all_ds):
        eval_hdl.set_curr_model(model)
        eval_hdl.set_curr_ds(ds)
        data[model][ds] = data_io.load_json(Path("..", eval_hdl.get_eval_path()))
    return data


eval_data = load_eval_responses()

In [5]:
emb_model = "mpnet"  # granite
samples, summaries = load_triples_data(emb_model=emb_model)

In [6]:
# Organise cosine similarity values
sample_cos = {model: {ds: {} for ds in ds_list} for model in model_list}
sample_avgs = {model: {ds: {} for ds in ds_list} for model in model_list}
for model, ds in zip(all_models, all_ds):
    for s, s_data in samples[model][ds].items():
        sample_cos[model][ds][s] = [e["sk_similar"]["cosine"] for e in s_data]
        sample_avgs[model][ds][s] = np.mean(sample_cos[model][ds][s]).item()

/opt/homebrew/Caskroom/miniforge/base/envs/mllm_diag/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/homebrew/Caskroom/miniforge/base/envs/mllm_diag/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [7]:
# Check the range of cosine similarity values obtained
for m, ds in zip(all_models, all_ds):
    print("=" * 70)
    print(f"Ranges for {m.upper()} on {ds.upper()}")
    print("\t\t = Min. = \t = Max. = \t = Avg. = \t Support")
    # First, get sample-level averages

    mes = [v if not np.isnan(v) else 0.0 for v in sample_avgs[m][ds].values()]
    min_mes = np.min(mes).item()
    max_mes = np.max(mes).item()
    avg_mes = np.mean(mes).item()
    print(
        f"Sample-level \t {min_mes:.4f} \t {max_mes:.4f} \t {avg_mes:.4f} \t {len(mes)}"
    )
    # Then, consider the individual measures for each triple
    mes = [e if not np.isnan(e) else 0.0 for t in sample_cos[m][ds].values() for e in t]
    min_mes = np.min(mes).item()
    max_mes = np.max(mes).item()
    print(f"Triple-level \t {min_mes:.4f} \t {max_mes:.4f} \t ------ \t {len(mes)}")

    print("-" * 70)
    negs = [m for m in mes if m <= 0]
    print(f"Num. negative triple similarities: {len(negs)}/{len(mes)}")
print("=" * 70)

Ranges for INTERNVL2 on LLAVA-BENCH
		 = Min. = 	 = Max. = 	 = Avg. = 	 Support
Sample-level 	 0.0502 	 0.6696 	 0.3030 	 58
Triple-level 	 -0.0531 	 0.9140 	 ------ 	 553
----------------------------------------------------------------------
Num. negative triple similarities: 173/553
Ranges for INTERNVL2 on MMBENCH
		 = Min. = 	 = Max. = 	 = Avg. = 	 Support
Sample-level 	 0.0552 	 0.8050 	 0.3262 	 139
Triple-level 	 -0.0592 	 1.0000 	 ------ 	 1233
----------------------------------------------------------------------
Num. negative triple similarities: 382/1233
Ranges for INTERNVL2 on SEED
		 = Min. = 	 = Max. = 	 = Avg. = 	 Support
Sample-level 	 0.0555 	 0.9559 	 0.3319 	 150
Triple-level 	 -0.0725 	 1.0000 	 ------ 	 1163
----------------------------------------------------------------------
Num. negative triple similarities: 268/1163
Ranges for INTERNVL2 on VQAV2
		 = Min. = 	 = Max. = 	 = Avg. = 	 Support
Sample-level 	 0.1308 	 0.9209 	 0.4435 	 150
Triple-level 	 -0.0477 	 1.

Data for these 3 samples is not enough to begin with. Let's skip these.

In [8]:
# Problematic values for Qwen2.5-VL + VQA.
# - ('120162001', nan) -> No values saved: []
# - ('333538005', nan) -> No values saved: []
# Problematic values for ShareGPT4V + SEED.
# - ('2358', nan) -> No values saved: []

# print(samples["qwen2_5_vl"]["vqav2"]["120162001"])
# print(samples["qwen2_5_vl"]["vqav2"]["333538005"])
# print(samples["sharegpt4v"]["seed"]["2358"])

## Pepare main dataframe

We have very few triple similarities that are slightly negative, let's clip those to `0.0` when creating the dataframe.

In [9]:
df = pd.DataFrame()
for model, ds in zip(all_models, all_ds):
    for q_idx in samples[model][ds]:
        # We have very few cosine similarities that are sligthly negative.
        # Clip those to 0.0. They are basically irrelevant.
        avg_cosine = (
            sample_avgs[model][ds][q_idx]
            if sample_avgs[model][ds][q_idx] > 0.0
            else 0.0
        )
        rows = []
        """
        Take care of empty measures
            - Qwen2.5-VL + VQAv2: '120162001', '333538005'
            - ShareGPT4V + SEED: '2358'
        """
        if len(samples[model][ds][q_idx]) > 0:
            for m in samples[model][ds][q_idx]:
                if m["sk_similar"].keys() & {"from_concept", "rel", "to_concept"}:
                    sk_triple = {
                        "from_concept": m["sk_similar"]["from_concept"],
                        "relationship": m["sk_similar"]["rel"],
                        "to_concept": m["sk_similar"]["to_concept"],
                    }
                else:
                    sk_triple = None

                rows.append(
                    {
                        "question_id": str(q_idx),
                        "model": model,
                        "dataset": ds,
                        "sk_triple": sk_triple,
                        "rk_triple": {
                            "from_concept": m["from_concept"],
                            "relationship": m["rel"],
                            "to_concept": m["to_concept"],
                        },
                        "triple_cosine": m["sk_similar"]["cosine"]
                        if m["sk_similar"]["cosine"] > 0.0
                        else 0.0,
                    }
                )
        else:
            rows.append(
                {
                    "question_id": str(q_idx),
                    "model": model,
                    "dataset": ds,
                    "sk_triple": None,
                    "rk_triple": None,
                    "triple_cosine": 0.0,
                }
            )

        temp_df = pd.DataFrame.from_records(rows)
        df = pd.concat([df, temp_df], ignore_index=True)

In [10]:
df.head()

,question_id,model,dataset,sk_triple,rk_triple,triple_cosine
0,0,internvl2,llava-bench,None,"{'from_concept': 'volcanic tuff cone', 'relati...",0.000000
1,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'grow...","{'from_concept': 'volcanic tuff cone', 'relati...",0.362474
2,0,internvl2,llava-bench,"{'from_concept': 'vegetation', 'relationship':...","{'from_concept': 'coastal area', 'relationship...",0.597013
3,0,internvl2,llava-bench,None,"{'from_concept': 'coastal area', 'relationship...",0.000000
4,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'on o...","{'from_concept': 'beach', 'relationship': 'is ...",0.212477


In [11]:
# Add sample_cosine

df["sample_cosine"] = 0.0

for m, ds in zip(all_models, all_ds):
    curr_df = df[(df["dataset"] == ds) & (df["model"] == m)]
    sample_ids = list(curr_df["question_id"].unique())
    for idx in sample_ids:
        curr_sample = curr_df[curr_df["question_id"] == idx]
        sample_sim = np.average(curr_sample["triple_cosine"]).item()
        df_row_ids = curr_sample.index  # These will be the ones to update
        df.loc[df_row_ids, "sample_cosine"] = sample_sim

In [12]:
df.head()

,question_id,model,dataset,sk_triple,rk_triple,triple_cosine,sample_cosine
0,0,internvl2,llava-bench,None,"{'from_concept': 'volcanic tuff cone', 'relati...",0.000000,0.286484
1,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'grow...","{'from_concept': 'volcanic tuff cone', 'relati...",0.362474,0.286484
2,0,internvl2,llava-bench,"{'from_concept': 'vegetation', 'relationship':...","{'from_concept': 'coastal area', 'relationship...",0.597013,0.286484
3,0,internvl2,llava-bench,None,"{'from_concept': 'coastal area', 'relationship...",0.000000,0.286484
4,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'on o...","{'from_concept': 'beach', 'relationship': 'is ...",0.212477,0.286484


In [13]:
total = len(df)
n_not_matched = len(df[df["sk_triple"].isnull()])
print(f"Total n. triples: {total}.")
print(f"N. not mached: {n_not_matched} ({(100 * n_not_matched / total):.1f}%).")

Total n. triples: 9446.
N. not mached: 1828 (19.4%).


Checking number of triples to make sure nothing got lost

In [14]:
# Check the number of triples in 'samples' vs 'df'
# Note that the following are correct since we have no similar triples
#   QWEN2_5_VL on VQAV2: 		 260 --> 262 	 Missing: -2
#   SHAREGPT4V on SEED: 		 230 --> 231 	 Missing: -1
for model, ds in zip(all_models, all_ds):
    df_count = len(df[(df["model"] == model) & (df["dataset"] == ds)])
    samples_count = len([m for t in sample_cos[model][ds].values() for m in t])
    print(
        f"{model.upper()} on {ds.upper()}: \t\t {samples_count} --> {df_count} \t Missing: {samples_count - df_count}"
    )

INTERNVL2 on LLAVA-BENCH: 		 553 --> 553 	 Missing: 0
INTERNVL2 on MMBENCH: 		 1233 --> 1233 	 Missing: 0
INTERNVL2 on SEED: 		 1163 --> 1163 	 Missing: 0
INTERNVL2 on VQAV2: 		 934 --> 934 	 Missing: 0
LLAVA-1.6 on LLAVA-BENCH: 		 265 --> 265 	 Missing: 0
LLAVA-1.6 on MMBENCH: 		 745 --> 745 	 Missing: 0
LLAVA-1.6 on SEED: 		 786 --> 786 	 Missing: 0
LLAVA-1.6 on VQAV2: 		 879 --> 879 	 Missing: 0
QWEN2_5_VL on LLAVA-BENCH: 		 446 --> 446 	 Missing: 0
QWEN2_5_VL on MMBENCH: 		 544 --> 544 	 Missing: 0
QWEN2_5_VL on SEED: 		 491 --> 491 	 Missing: 0
QWEN2_5_VL on VQAV2: 		 260 --> 262 	 Missing: -2
SHAREGPT4V on LLAVA-BENCH: 		 259 --> 259 	 Missing: 0
SHAREGPT4V on MMBENCH: 		 612 --> 612 	 Missing: 0
SHAREGPT4V on SEED: 		 230 --> 231 	 Missing: -1
SHAREGPT4V on VQAV2: 		 43 --> 43 	 Missing: 0


Add model performance to dataframe

In [15]:
# Data preparation
eval_data_clean = {model: {ds: [] for ds in eval_data[model]} for model in eval_data}
for model, ds in zip(all_models, all_ds):
    if ds in OPEN_DATASETS:
        ranges = list(eval_data[model][ds]["ranges"].keys())
        for r in ranges:
            eval_data_clean[model][ds].extend(
                [
                    {"question_id": int(d["question_id"]), "f1": d["measures"]["f1"]}
                    for d in eval_data[model][ds]["ranges"][r]
                ]
            )
        # Parse remaning, invalid responses
        eval_data_clean[model][ds].extend(
            [
                {"question_id": int(d["question_id"]), "f1": 0.0}
                for d in eval_data[model][ds]["invalid_resp"]
            ]
        )
    elif ds in MC_DATASETS:
        # Parse correct_resp and wrong_resp
        eval_data_clean[model][ds].extend(
            [
                {"question_id": int(d["question_id"]), "correct": 1}
                for d in eval_data[model][ds]["correct_resp"]
            ]
        )
        eval_data_clean[model][ds].extend(
            [
                {"question_id": int(d["question_id"]), "correct": 0}
                for d in eval_data[model][ds]["wrong_resp"]
            ]
        )
        # Parse remaning, invalid responses
        eval_data_clean[model][ds].extend(
            [
                {"question_id": int(d["question_id"]), "correct": 0}
                for d in eval_data[model][ds]["invalid_resp"]
            ]
        )
    else:
        print("Error")

In [16]:
def get_outcome(row, eval_data_clean):
    data = next(
        (
            e
            for e in eval_data_clean[row["model"]][row["dataset"]]
            if str(e["question_id"]) == row["question_id"]
        ),
        None,
    )
    return data


df["oe_outcome"] = df.apply(
    lambda x: (
        get_outcome(x, eval_data_clean)["f1"] if x["dataset"] in OPEN_DATASETS else None
    ),
    axis=1,
)
df["mc_outcome"] = df.apply(
    lambda x: (
        get_outcome(x, eval_data_clean)["correct"]
        if x["dataset"] in MC_DATASETS
        else None
    ),
    axis=1,
)
df.head()

,question_id,model,dataset,sk_triple,rk_triple,triple_cosine,sample_cosine,oe_outcome,mc_outcome
0,0,internvl2,llava-bench,None,"{'from_concept': 'volcanic tuff cone', 'relati...",0.000000,0.286484,0.792572,NaN
1,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'grow...","{'from_concept': 'volcanic tuff cone', 'relati...",0.362474,0.286484,0.792572,NaN
2,0,internvl2,llava-bench,"{'from_concept': 'vegetation', 'relationship':...","{'from_concept': 'coastal area', 'relationship...",0.597013,0.286484,0.792572,NaN
3,0,internvl2,llava-bench,None,"{'from_concept': 'coastal area', 'relationship...",0.000000,0.286484,0.792572,NaN
4,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'on o...","{'from_concept': 'beach', 'relationship': 'is ...",0.212477,0.286484,0.792572,NaN


In [17]:
df[df["dataset"] == "seed"].head()

,question_id,model,dataset,sk_triple,rk_triple,triple_cosine,sample_cosine,oe_outcome,mc_outcome
1786,3151,internvl2,seed,"{'from_concept': 'counter', 'relationship': 'i...","{'from_concept': 'bar counter', 'relationship'...",0.563839,0.368984,NaN,1.0
1787,3151,internvl2,seed,"{'from_concept': 'door', 'relationship': 'lead...","{'from_concept': 'bar counter', 'relationship'...",0.541770,0.368984,NaN,1.0
1788,3151,internvl2,seed,"{'from_concept': 'lamps', 'relationship': 'abo...","{'from_concept': 'bar counter', 'relationship'...",0.581275,0.368984,NaN,1.0
1789,3151,internvl2,seed,"{'from_concept': 'bottles', 'relationship': 's...","{'from_concept': 'bar counter', 'relationship'...",0.381331,0.368984,NaN,1.0
1790,3151,internvl2,seed,"{'from_concept': 'counter', 'relationship': 'b...","{'from_concept': 'bar counter', 'relationship'...",0.285953,0.368984,NaN,1.0


In [18]:
for model, ds in zip(all_models, all_ds):
    subdf = df[(df["model"] == model) & (df["dataset"] == ds)]
    print(f"{model.upper()} on {ds.upper()}: {len(subdf['question_id'].unique())}")

INTERNVL2 on LLAVA-BENCH: 58
INTERNVL2 on MMBENCH: 139
INTERNVL2 on SEED: 150
INTERNVL2 on VQAV2: 150
LLAVA-1.6 on LLAVA-BENCH: 44
LLAVA-1.6 on MMBENCH: 133
LLAVA-1.6 on SEED: 129
LLAVA-1.6 on VQAV2: 142
QWEN2_5_VL on LLAVA-BENCH: 56
QWEN2_5_VL on MMBENCH: 139
QWEN2_5_VL on SEED: 150
QWEN2_5_VL on VQAV2: 150
SHAREGPT4V on LLAVA-BENCH: 49
SHAREGPT4V on MMBENCH: 135
SHAREGPT4V on SEED: 109
SHAREGPT4V on VQAV2: 21


Current dataframe
- `question_id`
- `model`
- `dataset`
- `sample_cosine`: sample-level, avg similarity
- `sk_triple`
- `rk_triple`
- `triple_cosine`: triple-level, similarity
- `oe_outcome`: BERTScore F1
- `mc_outcome`: correct (1.0) or incorrect (0.0)

**Double-checking before proceeding**

In [ ]:
# Load clean data
def get_og_data(data_path: Path) -> dict:
    raw_data = data_io.load_pickle(data_path)
    clean_data = {}
    for id, graph in raw_data.items():
        q_idx = str(id)
        clean_data[q_idx] = []
        for rel in graph.edges:
            clean_data[q_idx].append(
                {
                    "from_concept": rel[0],
                    "rel": graph[rel[0]][rel[1]]["label"],
                    "to_concept": rel[1],
                }
            )
    return clean_data

In [ ]:
def compare_og_data(init_triple: dict, df_triples: list) -> bool:
    for triple in df_triples:
        if triple is None:
            return False
        if (
            init_triple["from_concept"] == triple["from_concept"]
            and init_triple["rel"] == triple["relationship"]
            and init_triple["to_concept"] == triple["to_concept"]
        ):
            return True
    return False

1. Check that all RKs are present (incl. ones w/o a SK paired)

In [ ]:
rk_hdl = RKHandler()
rk_data = {model: {ds: {} for ds in ds_list} for model in model_list}
# Load the initial rk data
for model, ds in zip(all_models, all_ds):
    rk_hdl.set_curr_model(model)
    rk_hdl.set_curr_ds(ds)
    curr_path = Path("..", rk_hdl.get_final_pkl_path())
    rk_data[model][ds] = get_og_data(curr_path)

In [ ]:
for model, ds in zip(all_models, all_ds):
    init_ids = list(rk_data[model][ds])
    df_ids = df[(df["model"] == model) & (df["dataset"] == ds)]["question_id"].unique()
    print(f"Checking {model} x {ds}")
    # Simple checks with question ids
    len_match = len(init_ids) == len(df_ids)
    q_ids_match = len(set(init_ids).difference(set(df_ids))) == 0
    if len_match and q_ids_match:
        print(" > All samples are present.")

    # Check that we actually have all triples
    missing = 0
    for q_idx in init_ids:
        df_rks = df[
            (df["model"] == model)
            & (df["dataset"] == ds)
            & (df["question_id"] == q_idx)
        ]["rk_triple"]
        for init_rk in rk_data[model][ds][q_idx]:
            if not compare_og_data(init_rk, df_rks):
                missing += 1
                print(f"Problem with {q_idx}: {init_rk}")
    print(f" > N. problems: {missing}")

**-> All RK samples (ids) and corresponding triples are present**

2. Check which, and how many, SKs are missing (some where not saved when RK < SK)

In [ ]:
sk_hdl = SKHandler()
sk_data = {ds: {} for ds in ds_list}
# Load the initial sk data
for ds in ds_list:
    sk_hdl.set_curr_ds(ds)
    for c in sk_hdl.get_classes():
        sk_hdl.set_curr_class(c)
        curr_path = Path("..", sk_hdl.get_exp_pkl_path())
        sk_data[ds].update(get_og_data(curr_path))

In [ ]:
# total_missing = {model: {ds: 0 for ds in ds_list} for model in model_list}
init_sk_counts = {ds: {} for ds in ds_list}
for ds in ds_list:
    init_sk_counts[ds].update({q_idx: len(sks) for q_idx, sks in sk_data[ds].items()})

for model, ds in zip(all_models, all_ds):
    sk_hdl.set_curr_ds(ds)

    # Simple checks with question ids
    # Expectations
    df_ids = df[(df["model"] == model) & (df["dataset"] == ds)]["question_id"].unique()
    init_ids = list(sk_data[ds].keys())

    # We don't have RKs for all samples -> Not all question_ids will be present.
    len_match = len(init_ids) >= len(df_ids)
    q_ids_match = len(set(init_ids).difference(set(df_ids))) >= 0
    if len_match and q_ids_match:
        print(f"{model} x {ds}: All samples are present.")
    else:
        print(f"{model} x {ds}: Problem.")

    # Find out which SK triples have not been used to compute similarity
    leftovers = {model: {ds: {} for ds in ds_list} for model in model_list}
    init_count = sum(init_sk_counts[ds][q_idx] for q_idx in df_ids)
    sks_in_df = df[
        (df["model"] == model) & (df["dataset"] == ds) & (df["sk_triple"].notnull())
    ]
    # total_missing = 0
    for q_idx in df_ids:
        leftovers[model][ds][q_idx] = []
        curr_sks = sks_in_df[sks_in_df["question_id"] == q_idx]["sk_triple"]
        init_sample = sk_data[ds][q_idx]

        for init_sk in init_sample:
            if not compare_og_data(init_sk, curr_sks):
                leftovers[model][ds][q_idx].append(init_sk)
                # total_missing += 1
    # total_missing[model][ds] += missing
    total_missing = len(
        [sk for sk_batch in leftovers[model][ds].values() for sk in sk_batch]
    )
    test = init_count == len(sks_in_df) + total_missing
    print(f"Checking {model} x {ds} --> {test}")
    print(f" - Initial SKs: {init_count}")
    print(f" - SKs in df: {len(sks_in_df)}")
    print(f" - SKs missing from df: {total_missing}")

In [ ]:
for q_idx, sks in leftovers["sharegpt4v"]["vqav2"].items():
    print(f"{q_idx} -> should have {len(sk_data['vqav2'][q_idx])}; missing {len(sks)}.")

**-> All the missing SKs are saved in leftovers and the counts add up**

### Add behaviour types

Strict split after manual inspection of the equal 3-way split

In [19]:
prec = 6

strict_split = {
    1: {"start": 0.75, "end": 1.0},
    2: {"start": 0.350000, "end": 0.749999},
    3: {"start": 0.0, "end": 0.349999},
}
print(strict_split)

{1: {'start': 0.75, 'end': 1.0}, 2: {'start': 0.35, 'end': 0.749999}, 3: {'start': 0.0, 'end': 0.349999}}


In [20]:
def get_behaviour_type(sim_val, ranges):
    temp = float(round(sim_val, prec))
    for idx, r in ranges.items():
        if temp >= r["start"] and temp <= r["end"]:
            return idx
    # Dirty fix for cosine values that go beyond 1.0 (for some reason)
    if temp >= 1.0:
        return 1
    print("Error; returning None")
    return None

In [21]:
analysis_type = "strict"
# This is straightforward and needed to classify each sample
df["triple_type"] = df.apply(
    lambda x: get_behaviour_type(x["triple_cosine"], strict_split), axis=1
)
df.head()

,question_id,model,dataset,sk_triple,rk_triple,triple_cosine,sample_cosine,oe_outcome,mc_outcome,triple_type
0,0,internvl2,llava-bench,None,"{'from_concept': 'volcanic tuff cone', 'relati...",0.000000,0.286484,0.792572,NaN,3
1,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'grow...","{'from_concept': 'volcanic tuff cone', 'relati...",0.362474,0.286484,0.792572,NaN,2
2,0,internvl2,llava-bench,"{'from_concept': 'vegetation', 'relationship':...","{'from_concept': 'coastal area', 'relationship...",0.597013,0.286484,0.792572,NaN,2
3,0,internvl2,llava-bench,None,"{'from_concept': 'coastal area', 'relationship...",0.000000,0.286484,0.792572,NaN,3
4,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'on o...","{'from_concept': 'beach', 'relationship': 'is ...",0.212477,0.286484,0.792572,NaN,3


In [22]:
def get_sample_type(sample_rows, ranges):
    """
    First implementation: check sizes and do average of cosine similarities
    """
    sample_cosine = sample_rows["sample_cosine"].iloc[0]
    # If the average similarity is within the lowest range, simply return
    if sample_cosine >= ranges[3]["start"] and sample_cosine <= ranges[3]["end"]:
        return (3, False)
    # Otherwise, have a look at the number of SKs and RKs that we have
    count_rk = len(sample_rows) - sample_rows["rk_triple"].isnull().sum()
    count_sk = len(sample_rows) - sample_rows["sk_triple"].isnull().sum()

    if count_rk <= count_sk:
        # Here, we might have type 1 behaviours
        if sample_cosine >= ranges[1]["start"] and sample_cosine <= ranges[1]["end"]:
            return (1, False)
        else:
            # Here, the similarity can't be from type 3 -- has to be type 2.
            # Let's ignore size for now and classify as type 2.
            return (2, True)
    else:
        # Here, we might have type 2 behaviours
        if sample_cosine >= ranges[2]["start"] and sample_cosine <= ranges[2]["end"]:
            return (2, False)
        else:
            # Here, the similarity can't be from type 3 -- has to be type 1.
            # Let's ignore size for now and classify as type 1.
            return (1, True)


# Init 'sample_type' col
df["sample_type"] = 0
df["slack"] = False

for m, ds in zip(all_models, all_ds):
    curr_df = df[(df["dataset"] == ds) & (df["model"] == m)]
    sample_ids = list(curr_df["question_id"].unique())
    for idx in sample_ids:
        curr_sample = curr_df[curr_df["question_id"] == idx]
        df_row_ids = curr_sample.index  # These will be the ones to update
        sample_info = get_sample_type(curr_sample, strict_split)
        df["sample_type"].loc[df_row_ids] = sample_info[0]
        df["slack"].loc[df_row_ids] = sample_info[1]


# df["sample_type"] = df.apply(
#     lambda x: get_behaviour_type(x["sample_cosine"], strict_split), axis=1
# )

/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_56484/2473757877.py:42: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["sample_type"].loc[df_row_ids] = sample_info[0]
/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_56

In [23]:
df.head()

,question_id,model,dataset,sk_triple,rk_triple,triple_cosine,sample_cosine,oe_outcome,mc_outcome,triple_type,sample_type,slack
0,0,internvl2,llava-bench,None,"{'from_concept': 'volcanic tuff cone', 'relati...",0.000000,0.286484,0.792572,NaN,3,3,False
1,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'grow...","{'from_concept': 'volcanic tuff cone', 'relati...",0.362474,0.286484,0.792572,NaN,2,3,False
2,0,internvl2,llava-bench,"{'from_concept': 'vegetation', 'relationship':...","{'from_concept': 'coastal area', 'relationship...",0.597013,0.286484,0.792572,NaN,2,3,False
3,0,internvl2,llava-bench,None,"{'from_concept': 'coastal area', 'relationship...",0.000000,0.286484,0.792572,NaN,3,3,False
4,0,internvl2,llava-bench,"{'from_concept': 'tree', 'relationship': 'on o...","{'from_concept': 'beach', 'relationship': 'is ...",0.212477,0.286484,0.792572,NaN,3,3,False


In [24]:
# Check n samples with slack for classification
for model, ds in zip(all_models, all_ds):
    curr = df[(df["model"] == model) & (df["dataset"] == ds)]
    sample_ids = curr["question_id"].unique()
    count = len(curr[curr["slack"]]["question_id"].unique())
    print(f"{model} + {ds} --> {count} ({100 * count / len(sample_ids):.2f}%)")

internvl2 + llava-bench --> 17 (29.31%)
internvl2 + mmbench --> 35 (25.18%)
internvl2 + seed --> 42 (28.00%)
internvl2 + vqav2 --> 80 (53.33%)
llava-1.6 + llava-bench --> 20 (45.45%)
llava-1.6 + mmbench --> 62 (46.62%)
llava-1.6 + seed --> 60 (46.51%)
llava-1.6 + vqav2 --> 78 (54.93%)
qwen2_5_vl + llava-bench --> 13 (23.21%)
qwen2_5_vl + mmbench --> 92 (66.19%)
qwen2_5_vl + seed --> 66 (44.00%)
qwen2_5_vl + vqav2 --> 84 (56.00%)
sharegpt4v + llava-bench --> 29 (59.18%)
sharegpt4v + mmbench --> 73 (54.07%)
sharegpt4v + seed --> 75 (68.81%)
sharegpt4v + vqav2 --> 15 (71.43%)


**Sample some example for each behaviour type for manual checking.**

In [ ]:
df.head()

In [ ]:
group_counts = {model: {ds: None for ds in ds_list} for model in model_list}

out_path = Path("group_counts", "full")
data_io.make_dir(out_path)

for model, ds in zip(all_models, all_ds):
    group_counts[model][ds] = (
        df[(df["model"] == model) & (df["dataset"] == ds)]
        .groupby(["question_id", "sample_type"])
        .count()
    )
    group_counts[model][ds].drop(
        [
            "dataset",
            "sk_triple",
            "rk_triple",
            "triple_cosine",
            "sample_cosine",
            "oe_outcome",
            "mc_outcome",
            "triple_type",
            "slack",
        ],
        axis=1,
        inplace=True,
    )
    group_counts[model][ds].rename({"model": "n_triples"}, axis=1, inplace=True)
    out_file = out_path.joinpath(f"counts_{model}_{ds}.csv")
    group_counts[model][ds].to_csv(out_file)

In [ ]:
for model, ds in zip(all_models, all_ds):
    print(f"{model} + {ds} info")
    display(group_counts[model][ds].groupby(["sample_type"]).count())

In [ ]:
group_counts["internvl2"]["llava-bench"].query("sample_type == 3").head()

In [ ]:
for model, ds in zip(all_models, all_ds):
    curr_df = group_counts[model][ds]
    for type in strict_split.keys():
        out_path = Path("examples", model, ds, f"type_{type}")
        data_io.make_dir(out_path)

        subset = curr_df.query(f"sample_type == {type}")
        if len(subset) == 0:
            ids_subset = []
        else:
            sample_size = 5
            if len(subset) < sample_size:
                sample_size = len(subset)
            ids_subset = subset.sample(sample_size, replace=False)
            ids_subset.reset_index(inplace=True)
            ids_subset = ids_subset["question_id"].to_list()
        # print(ids_subset)

        # Get from main df
        for idx in ids_subset:
            to_save = pd.DataFrame()
            to_save = pd.concat(
                [
                    to_save,
                    df[
                        (df["model"] == model)
                        & (df["dataset"] == ds)
                        & (df["question_id"] == idx)
                    ],
                ],
                ignore_index=True,
            )
            out_file = out_path.joinpath(f"{idx}.csv")
            to_save.to_csv(out_file)

In [ ]:
sample_ids = df[(df["dataset"] == "llava-bench") & (df["model"] == "internvl2")][
    "question_id"
].unique()
print(len(sample_ids))

c_slack = len(
    df[(df["dataset"] == "llava-bench") & (df["model"] == "internvl2") & (df["slack"])][
        "question_id"
    ].unique()
)
print(c_slack)

c_base = len(
    df[
        (df["dataset"] == "llava-bench") & (df["model"] == "internvl2") & (~df["slack"])
    ]["question_id"].unique()
)
print(c_base)

Plotting behaviours counts wrt individual triples and samples

In [25]:
def get_counts(df, ranges, aggr="sample"):
    if aggr == "model":
        counts = {m: {ds: {} for ds in ds_list} for m in model_list}
    elif aggr == "type":
        counts = {idx: {ds: [] for ds in ds_list} for idx in ranges}
    elif aggr == "sample":
        counts = {
            idx: {ds: {"base": [], "slack": []} for ds in ds_list} for idx in ranges
        }
    else:
        print("aggregation not recognised")
        return

    for idx in ranges.keys():
        for m, ds in zip(all_models, all_ds):
            curr_subset = df[(df["dataset"] == ds) & (df["model"] == m)]
            if aggr == "type":
                count = len(curr_subset[(curr_subset["triple_type"] == idx)])
            else:
                count_base = len(
                    curr_subset[
                        (~curr_subset["slack"]) & (curr_subset["sample_type"] == idx)
                    ]["question_id"].unique()
                )
                count_slack = len(
                    curr_subset[
                        (curr_subset["slack"]) & (curr_subset["sample_type"] == idx)
                    ]["question_id"].unique()
                )

            if aggr == "model":
                counts[m][ds][idx] = count
            elif aggr == "type":
                counts[idx][ds].append(count)
            elif aggr == "sample":
                counts[idx][ds]["base"].append(count_base)
                counts[idx][ds]["slack"].append(count_slack)
            else:
                print("aggregation not recognised")
                return

    return counts

In [26]:
counts_sample = get_counts(df, strict_split, aggr="sample")
counts_sample

{1: {'llava-bench': {'base': [0, 0, 0, 0], 'slack': [0, 0, 0, 0]},
  'mmbench': {'base': [1, 13, 10, 16], 'slack': [0, 0, 0, 0]},
  'seed': {'base': [2, 3, 10, 11], 'slack': [0, 0, 0, 0]},
  'vqav2': {'base': [12, 5, 24, 5], 'slack': [0, 0, 0, 0]}},
 2: {'llava-bench': {'base': [2, 1, 2, 0], 'slack': [17, 20, 13, 29]},
  'mmbench': {'base': [19, 3, 8, 3], 'slack': [35, 62, 92, 73]},
  'seed': {'base': [10, 9, 2, 0], 'slack': [42, 60, 66, 75]},
  'vqav2': {'base': [10, 11, 0, 0], 'slack': [80, 78, 84, 15]}},
 3: {'llava-bench': {'base': [39, 23, 41, 20], 'slack': [0, 0, 0, 0]},
  'mmbench': {'base': [84, 55, 29, 43], 'slack': [0, 0, 0, 0]},
  'seed': {'base': [96, 57, 72, 23], 'slack': [0, 0, 0, 0]},
  'vqav2': {'base': [48, 48, 42, 1], 'slack': [0, 0, 0, 0]}}}

In [ ]:
if analysis_type:
    out_path = Path(CHARTS_DIR, f"overlap_samples_{analysis_type}_{emb_model}.pdf")
else:
    out_path = Path(CHARTS_DIR, f"overlap_samples_{emb_model}.pdf")

vis.plot_sample_behaviours(counts_sample, out_path, split=False)

In [ ]:
out_path = Path(CHARTS_DIR, f"overlap_samples_stack_{emb_model}.pdf")
vis.plot_sample_behaviours(counts_sample, out_path, split=True)

In [29]:
counts_type = get_counts(df, strict_split, aggr="type")
counts_type

{1: {'llava-bench': [16, 13, 10, 13],
  'mmbench': [103, 66, 86, 62],
  'seed': [74, 49, 28, 23],
  'vqav2': [113, 98, 52, 8]},
 2: {'llava-bench': [199, 116, 100, 106],
  'mmbench': [431, 287, 288, 287],
  'seed': [385, 289, 159, 132],
  'vqav2': [355, 338, 118, 25]},
 3: {'llava-bench': [338, 136, 336, 140],
  'mmbench': [699, 392, 170, 263],
  'seed': [704, 448, 304, 76],
  'vqav2': [466, 443, 92, 10]}}

In [ ]:
if analysis_type:
    out_path = Path(CHARTS_DIR, f"overlap_types_{analysis_type}_{emb_model}.pdf")
else:
    out_path = Path(CHARTS_DIR, f"overlap_types_{emb_model}.pdf")

vis.plot_triple_behaviours(counts_type, out_path)

In [31]:
# count_rk = len(sample_rows) - sample_rows["rk_triple"].isnull().sum()
# count_sk = len(sample_rows) - sample_rows["sk_triple"].isnull().sum()

diffs = {model: {ds: {} for ds in ds_list} for model in model_list}

for model, ds in zip(all_models, all_ds):
    curr_df = df[(df["model"] == model) & (df["dataset"] == ds)]
    sample_ids = curr_df["question_id"].unique()
    c = 0
    for idx in sample_ids:
        sample_rows = curr_df[curr_df["question_id"] == idx]
        diffs[model][ds][idx] = {
            "rks": len(sample_rows) - sample_rows["rk_triple"].isnull().sum(),
            "sks": len(sample_rows) - sample_rows["sk_triple"].isnull().sum(),
        }
        diffs[model][ds][idx]["diff"] = abs(
            diffs[model][ds][idx]["rks"] - diffs[model][ds][idx]["sks"]
        )
        # Cleanup
        diffs[model][ds][idx]["sks"] = diffs[model][ds][idx]["sks"].item()
        diffs[model][ds][idx]["rks"] = diffs[model][ds][idx]["rks"].item()
        diffs[model][ds][idx]["diff"] = diffs[model][ds][idx]["diff"].item()

        c += diffs[model][ds][idx]["diff"]

    print(f"{model} + {ds} --> {c} leftover triples")

internvl2 + llava-bench --> 169 leftover triples
internvl2 + mmbench --> 374 leftover triples
internvl2 + seed --> 262 leftover triples
internvl2 + vqav2 --> 158 leftover triples
llava-1.6 + llava-bench --> 46 leftover triples
llava-1.6 + mmbench --> 159 leftover triples
llava-1.6 + seed --> 143 leftover triples
llava-1.6 + vqav2 --> 141 leftover triples
qwen2_5_vl + llava-bench --> 129 leftover triples
qwen2_5_vl + mmbench --> 37 leftover triples
qwen2_5_vl + seed --> 50 leftover triples
qwen2_5_vl + vqav2 --> 0 leftover triples
sharegpt4v + llava-bench --> 53 leftover triples
sharegpt4v + mmbench --> 99 leftover triples
sharegpt4v + seed --> 5 leftover triples
sharegpt4v + vqav2 --> 0 leftover triples


In [32]:
diffs

{'internvl2': {'llava-bench': {'0': {'rks': 12, 'sks': 8, 'diff': 4},
   '1': {'rks': 13, 'sks': 10, 'diff': 3},
   '2': {'rks': 10, 'sks': 8, 'diff': 2},
   '3': {'rks': 9, 'sks': 7, 'diff': 2},
   '4': {'rks': 6, 'sks': 6, 'diff': 0},
   '5': {'rks': 7, 'sks': 7, 'diff': 0},
   '6': {'rks': 8, 'sks': 8, 'diff': 0},
   '7': {'rks': 5, 'sks': 5, 'diff': 0},
   '8': {'rks': 12, 'sks': 7, 'diff': 5},
   '9': {'rks': 9, 'sks': 3, 'diff': 6},
   '10': {'rks': 15, 'sks': 8, 'diff': 7},
   '11': {'rks': 4, 'sks': 4, 'diff': 0},
   '12': {'rks': 6, 'sks': 6, 'diff': 0},
   '13': {'rks': 6, 'sks': 6, 'diff': 0},
   '14': {'rks': 8, 'sks': 4, 'diff': 4},
   '15': {'rks': 6, 'sks': 3, 'diff': 3},
   '16': {'rks': 7, 'sks': 7, 'diff': 0},
   '17': {'rks': 12, 'sks': 7, 'diff': 5},
   '18': {'rks': 10, 'sks': 7, 'diff': 3},
   '19': {'rks': 10, 'sks': 8, 'diff': 2},
   '20': {'rks': 12, 'sks': 5, 'diff': 7},
   '21': {'rks': 9, 'sks': 3, 'diff': 6},
   '22': {'rks': 5, 'sks': 3, 'diff': 2},
   '23

Percentages of behaviours allocated

In [34]:
# Take out sample type, respecting the model and dataset ids
split_counts = {}

for m, ds in zip(all_models, all_ds):
    curr_df = df[(df["model"] == m) & (df["dataset"] == ds)]
    curr_types = (
        curr_df[["question_id", "sample_type"]]
        .drop_duplicates()["sample_type"]
        .to_list()
    )
    for ct in curr_types:
        split_counts[ct] = split_counts.get(ct, 0) + 1

total = sum(split_counts.values())
for behaviour_type, behaviour_count in split_counts.items():
    print(f"{behaviour_type}: {behaviour_count} ({100 * behaviour_count / total:.2f}%)")

3: 721 (41.11%)
2: 921 (52.51%)
1: 112 (6.39%)


# Correlation between behaviour similarity values and model performance 

Do using `sample_cosine`

In [35]:
corr_input = {m: {ds: {} for ds in ds_list} for m in model_list}
for model, ds in zip(all_models, all_ds):
    subdf = df[(df["model"] == model) & (df["dataset"] == ds)]
    corr_input[model][ds]["cosine"] = subdf["sample_cosine"].to_list()
    if ds in MC_DATASETS:
        corr_input[model][ds]["outcome"] = subdf["mc_outcome"].to_list()
    elif ds in OPEN_DATASETS:
        corr_input[model][ds]["outcome"] = subdf["oe_outcome"].to_list()
    else:
        print("Error")

Visualising the data

In [ ]:
# import matplotlib.pyplot as plt

# out_scatter = Path("scatter_plots")
# data_io.make_dir(out_scatter)

# for model, ds in zip(all_models, all_ds):
#     plt.scatter(
#         corr_input[model][ds]["cosine"],
#         corr_input[model][ds]["outcome"],
#         alpha=0.6,
#     )
#     plt.xlabel("Cosine similarity")
#     if ds in OPEN_DATASETS:
#         plt.ylabel("BERTScore-F1")
#     else:
#         plt.ylabel("Accuracy")
#     plt.title(f"{model} x {ds}")
#     plt.savefig(out_scatter.joinpath(f"{model}_{ds}.pdf"))
#     plt.close()

**Mutual Information**
- Is higher than 0 in general
- This implementation uses kNN and measures MI with natural logarithms and it increases very slowly
- MI > 2 might be uncommon in practice
- Problem: our sample sizes are kinda small... need to double-check for possible biases and overestimations (since we have values that are very high)

In [36]:
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif


def bootstrap_mi(X, Y, k, ds, n_boot=500, do_perm=False, random_state=0):
    rng = np.random.RandomState(random_state)
    n = len(X)
    mi_boot = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        Xb, Yb = X[idx], Y[idx]
        if do_perm:
            rng.shuffle(Yb)

        if ds in ["seed", "vqav2"]:
            mi_boot.append(
                mutual_info_classif(
                    Xb, Yb, n_neighbors=k, random_state=random_state
                ).item()
            )
        else:
            mi_boot.append(
                mutual_info_regression(
                    Xb, Yb, n_neighbors=k, random_state=random_state
                ).item()
            )

    mi_boot = np.array(mi_boot)
    mean = np.mean(mi_boot)
    low, high = np.percentile(mi_boot, [2.5, 97.5])  # <- 95% CI, 2 sides
    return mean.item(), low.item(), high.item()

In [37]:
k_vals = [3, 5, 6, 7, 8, 9, 10, 15, 20, 30]
n_boot = 1000
mi_data = {
    model: {
        ds: {
            "means_obs": [],
            "lows_obs": [],
            "highs_obs": [],
            "means_perm": [],
            "lows_perm": [],
            "highs_perm": [],
            "pvals": [],
            "mi_corrected": [],
        }
        for ds in ds_list
    }
    for model in model_list
}

for model, ds in zip(all_models, all_ds):
    print(f"{model.upper()} on {ds.upper()}")
    x = np.array(corr_input[model][ds]["cosine"]).reshape(-1, 1)
    y = np.array(corr_input[model][ds]["outcome"])
    for k in k_vals:
        m_obs, lo_obs, hi_obs = bootstrap_mi(x, y, k, ds, n_boot=n_boot)
        m_perm, lo_perm, hi_perm = bootstrap_mi(
            x, y, k, ds, n_boot=n_boot, do_perm=True
        )
        pval = (np.sum(m_perm >= m_obs) + 1) / (n_boot + 1)
        # Bias correction
        mi_corr = m_obs - m_perm

        # Update data
        mi_data[model][ds]["means_obs"].append(m_obs)
        mi_data[model][ds]["lows_obs"].append(lo_obs)
        mi_data[model][ds]["highs_obs"].append(hi_obs)
        mi_data[model][ds]["means_perm"].append(m_perm)
        mi_data[model][ds]["lows_perm"].append(lo_perm)
        mi_data[model][ds]["highs_perm"].append(hi_perm)
        mi_data[model][ds]["pvals"].append(pval)
        mi_data[model][ds]["mi_corrected"].append(mi_corr)

INTERNVL2 on LLAVA-BENCH
INTERNVL2 on MMBENCH
INTERNVL2 on SEED
INTERNVL2 on VQAV2
LLAVA-1.6 on LLAVA-BENCH
LLAVA-1.6 on MMBENCH
LLAVA-1.6 on SEED
LLAVA-1.6 on VQAV2
QWEN2_5_VL on LLAVA-BENCH
QWEN2_5_VL on MMBENCH
QWEN2_5_VL on SEED
QWEN2_5_VL on VQAV2
SHAREGPT4V on LLAVA-BENCH
SHAREGPT4V on MMBENCH
SHAREGPT4V on SEED
SHAREGPT4V on VQAV2


Print MI

In [38]:
print("\t\t\t = K =\t= MI corr =\t= Perm p-value =")
for model, ds in zip(all_models, all_ds):
    print(f"{model.upper()} + {ds.upper()}")
    for k, mean, p in zip(
        k_vals, mi_data[model][ds]["mi_corrected"], mi_data[model][ds]["pvals"]
    ):
        print(f"\t\t\t  {k} \t {mean:.3f} \t\t {p:.3f}")

			 = K =	= MI corr =	= Perm p-value =
INTERNVL2 + LLAVA-BENCH
			  3 	 3.864 		 0.001
			  5 	 3.577 		 0.001
			  6 	 3.367 		 0.001
			  7 	 3.129 		 0.001
			  8 	 2.869 		 0.001
			  9 	 2.601 		 0.001
			  10 	 2.328 		 0.001
			  15 	 1.214 		 0.001
			  20 	 0.731 		 0.001
			  30 	 0.394 		 0.001
INTERNVL2 + MMBENCH
			  3 	 3.734 		 0.001
			  5 	 3.438 		 0.001
			  6 	 3.200 		 0.001
			  7 	 2.918 		 0.001
			  8 	 2.615 		 0.001
			  9 	 2.318 		 0.001
			  10 	 2.031 		 0.001
			  15 	 1.105 		 0.001
			  20 	 0.746 		 0.001
			  30 	 0.392 		 0.001
INTERNVL2 + SEED
			  3 	 0.458 		 0.001
			  5 	 0.410 		 0.001
			  6 	 0.379 		 0.001
			  7 	 0.346 		 0.001
			  8 	 0.311 		 0.001
			  9 	 0.277 		 0.001
			  10 	 0.247 		 0.001
			  15 	 0.150 		 0.001
			  20 	 0.117 		 0.001
			  30 	 0.066 		 0.001
INTERNVL2 + VQAV2
			  3 	 0.593 		 0.001
			  5 	 0.518 		 0.001
			  6 	 0.476 		 0.001
			  7 	 0.434 		 0.001
			  8 	 0.394 		 0.001
			  9 	 0.359 		 0.001
			  1

Plot MI

In [ ]:
# Prepare for saving charts
out_path = Path("mi_charts")
data_io.make_dir(out_path)

In [ ]:
DS_MAP = {
    "llava-bench": "LLaVa-Bench",
    "mmbench": "MMBench",
    "seed": "SEED-Bench 2",
    "vqav2": "VQA v2",
}

MODEL_MAP = {
    "internvl2": "InternVL2",
    "llava-1.6": "LLaVa-1.6",
    "qwen2_5_vl": "Qwen2.5-VL",
    "sharegpt4v": "ShareGPT4V",
}

In [ ]:
# To match seaborn but it should still be legible with the default 10
# plt.rcParams["font.size"] = 18

for model, ds in zip(all_models, all_ds):
    plt.figure(figsize=(7, 4))
    # mi_data[model][ds]["means_obs"]
    plt.plot(
        k_vals,
        mi_data[model][ds]["mi_corrected"],
        "o-",
        label="Corrected MI",
        color="tab:blue",
    )
    plt.fill_between(
        k_vals,
        mi_data[model][ds]["lows_obs"],
        mi_data[model][ds]["highs_obs"],
        color="tab:blue",
        alpha=0.3,
    )

    # Annotate p-values above each point
    # mi_data[model][ds]["means_obs"]
    for k, mean, p in zip(
        k_vals, mi_data[model][ds]["mi_corrected"], mi_data[model][ds]["pvals"]
    ):
        plt.text(k, mean + 0.05, f"p={p:.3f}", ha="center", va="bottom", fontsize=8)

    plt.plot(
        k_vals,
        mi_data[model][ds]["means_perm"],
        "o--",
        label="Permutation baseline",
        color="tab:orange",
    )

    plt.fill_between(
        k_vals,
        mi_data[model][ds]["lows_perm"],
        mi_data[model][ds]["highs_perm"],
        color="tab:orange",
        alpha=0.3,
    )

    plt.xlabel("Number of neighbors (k)")
    plt.ylabel("Bootstrap MI (nats)")
    plt.title(f"{MODEL_MAP[model]} on {DS_MAP[ds]}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    # plt.show()

    out_file = out_path.joinpath(f"mi_{model}_{ds}.pdf")
    plt.savefig(out_file)

**Notes on the outcomes**
- *p-values* after permutations might indicate significance
- Bootstrapped *confidence intervals* seem alright
  - Lot of uncertainty for small subsets (e.g., sharegpt4v on vqav2)
- Analysed measures while varying the number of neighbours k
  - MI decreases as the increase k, which is good 
  - k=3 (default): MI is probably biased and inflated
  - k=10 : steepest decline (sometimes)
  - The 'correct' MI estimate should be somewhere between k=5 and k=10
  - Even the bias corrected MI are not that far off

**To report**
- Median (corrected) MI between k=5 and k=10
- CI for MI

In [40]:
final_range = list(range(1, 4))
final_k_vals = k_vals[1:4]
for model, ds in zip(all_models, all_ds):
    # Get median MI
    # final_mi = np.array(mi_data[model][ds]["means_obs"])[final_range]
    final_mi = np.array(mi_data[model][ds]["mi_corrected"])[final_range]
    final_mi = np.median(final_mi).item()
    # Get corresponding index in 'main' list
    # final_idx = np.where(np.array(mi_data[model][ds]["means_obs"]) == final_mi)[
    #     0
    # ].item()
    final_idx = np.where(np.array(mi_data[model][ds]["mi_corrected"]) == final_mi)[
        0
    ].item()
    # Get confidence interval for it
    final_low = mi_data[model][ds]["lows_obs"][final_idx]
    final_high = mi_data[model][ds]["highs_obs"][final_idx]

    print(
        f"{model.upper()} + {ds}: {final_mi:.3f} -- CI=({final_low:.3f}, {final_high:.3f})"
    )

INTERNVL2 + llava-bench: 3.367 -- CI=(3.158, 3.589)
INTERNVL2 + mmbench: 3.200 -- CI=(3.053, 3.350)
INTERNVL2 + seed: 0.379 -- CI=(0.325, 0.437)
INTERNVL2 + vqav2: 0.476 -- CI=(0.433, 0.526)
LLAVA-1.6 + llava-bench: 2.389 -- CI=(2.052, 2.711)
LLAVA-1.6 + mmbench: 2.512 -- CI=(2.333, 2.699)
LLAVA-1.6 + seed: 0.438 -- CI=(0.385, 0.497)
LLAVA-1.6 + vqav2: 0.481 -- CI=(0.437, 0.531)
QWEN2_5_VL + llava-bench: 3.008 -- CI=(2.764, 3.270)
QWEN2_5_VL + mmbench: 1.561 -- CI=(1.331, 1.792)
QWEN2_5_VL + seed: 0.188 -- CI=(0.115, 0.275)
QWEN2_5_VL + vqav2: 0.147 -- CI=(0.082, 0.230)
SHAREGPT4V + llava-bench: 2.058 -- CI=(1.737, 2.405)
SHAREGPT4V + mmbench: 2.116 -- CI=(1.932, 2.318)
SHAREGPT4V + seed: 0.292 -- CI=(0.198, 0.404)
SHAREGPT4V + vqav2: 0.558 -- CI=(0.139, 1.074)


**Outcome**
- On open-ended VQA datasets, we find strong dependency between model performance and the measured similarities
- On multiple-choice VQA datasets, we find weak to moderate dependency between model perfomance and the measured similarities

Re-do using `sample_type`

> Probaly won't use. Discretising X makes us loose so much info...

In [ ]:
corr_input = {m: {ds: {} for ds in ds_list} for m in model_list}
for model, ds in zip(all_models, all_ds):
    subdf = df[(df["model"] == model) & (df["dataset"] == ds)]
    corr_input[model][ds]["cosine"] = subdf["sample_type"].to_list()
    if ds in MC_DATASETS:
        corr_input[model][ds]["outcome"] = subdf["mc_outcome"].to_list()
    elif ds in OPEN_DATASETS:
        corr_input[model][ds]["outcome"] = subdf["oe_outcome"].to_list()
    else:
        print("Error")

In [ ]:
from sklearn.metrics import mutual_info_score


def bootstrap_mi_discrete(X, Y, k, ds, n_boot=500, do_perm=False, random_state=0):
    rng = np.random.RandomState(random_state)
    n = len(X)
    mi_boot = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        Xb, Yb = np.array(X[idx]).reshape(-1, 1), np.array(Y[idx]).reshape(-1, 1)
        # print(Xb.shape)
        # print(Yb.shape)

        if do_perm:
            rng.shuffle(Yb)

        if ds in ["seed", "vqav2"]:
            mi_boot.append(mutual_info_score(Xb.ravel(), Yb.ravel()))
        else:
            mi_boot.append(
                mutual_info_classif(
                    Yb, Xb.ravel(), n_neighbors=k, random_state=random_state
                ).item()
            )

    mi_boot = np.array(mi_boot)
    mean = np.mean(mi_boot)
    low, high = np.percentile(mi_boot, [2.5, 97.5])  # <- 95% CI, 2 sides
    return mean.item(), low.item(), high.item()


k_vals = [3, 5, 6, 7, 8, 9, 10, 15, 20, 30]
n_boot = 1000
mi_data = {
    model: {
        ds: {
            "means_obs": [],
            "lows_obs": [],
            "highs_obs": [],
            "means_perm": [],
            "lows_perm": [],
            "highs_perm": [],
            "pvals": [],
            "mi_corrected": [],
        }
        for ds in ds_list
    }
    for model in model_list
}

for model, ds in zip(all_models, all_ds):
    print(f"{model.upper()} on {ds.upper()}")
    x = np.array(corr_input[model][ds]["cosine"]).reshape(-1, 1)
    y = np.array(corr_input[model][ds]["outcome"])
    for k in k_vals:
        m_obs, lo_obs, hi_obs = bootstrap_mi_discrete(x, y, k, ds, n_boot=n_boot)
        m_perm, lo_perm, hi_perm = bootstrap_mi_discrete(
            x, y, k, ds, n_boot=n_boot, do_perm=True
        )
        pval = (np.sum(m_perm >= m_obs) + 1) / (n_boot + 1)
        # Bias correction
        mi_corr = m_obs - m_perm

        # Update data
        mi_data[model][ds]["means_obs"].append(m_obs)
        mi_data[model][ds]["lows_obs"].append(lo_obs)
        mi_data[model][ds]["highs_obs"].append(hi_obs)
        mi_data[model][ds]["means_perm"].append(m_perm)
        mi_data[model][ds]["lows_perm"].append(lo_perm)
        mi_data[model][ds]["highs_perm"].append(hi_perm)
        mi_data[model][ds]["pvals"].append(pval)
        mi_data[model][ds]["mi_corrected"].append(mi_corr)

In [ ]:
print("\t\t\t = K =\t= MI corr =\t= Perm p-value =")
for model, ds in zip(all_models, all_ds):
    print(f"{model.upper()} + {ds.upper()}")
    for k, mean, p in zip(
        k_vals, mi_data[model][ds]["mi_corrected"], mi_data[model][ds]["pvals"]
    ):
        print(f"\t\t\t  {k} \t {mean:.3f} \t\t {p:.3f}")

In [ ]:
# Prepare for saving charts
out_path = Path("mi_charts_type")
data_io.make_dir(out_path)

In [ ]:
for model, ds in zip(all_models, all_ds):
    plt.figure(figsize=(7, 4))
    # mi_data[model][ds]["means_obs"]
    plt.plot(
        k_vals,
        mi_data[model][ds]["mi_corrected"],
        "o-",
        label="Corrected MI",
        color="tab:blue",
    )
    plt.fill_between(
        k_vals,
        mi_data[model][ds]["lows_obs"],
        mi_data[model][ds]["highs_obs"],
        color="tab:blue",
        alpha=0.3,
    )

    # Annotate p-values above each point
    # mi_data[model][ds]["means_obs"]
    for k, mean, p in zip(
        k_vals, mi_data[model][ds]["mi_corrected"], mi_data[model][ds]["pvals"]
    ):
        plt.text(k, mean + 0.05, f"p={p:.3f}", ha="center", va="bottom", fontsize=8)

    plt.plot(
        k_vals,
        mi_data[model][ds]["means_perm"],
        "o--",
        label="Permutation baseline",
        color="tab:orange",
    )

    plt.fill_between(
        k_vals,
        mi_data[model][ds]["lows_perm"],
        mi_data[model][ds]["highs_perm"],
        color="tab:orange",
        alpha=0.3,
    )

    plt.xlabel("Number of neighbors (k)")
    plt.ylabel("Bootstrap MI (nats)")
    plt.title(f"{MODEL_MAP[model]} on {DS_MAP[ds]}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    # plt.show()

    out_file = out_path.joinpath(f"mi_{model}_{ds}.pdf")
    plt.savefig(out_file)

In [ ]:
final_range = list(range(1, 4))
final_k_vals = k_vals[1:4]
for model, ds in zip(all_models, all_ds):
    # Get median MI
    # final_mi = np.array(mi_data[model][ds]["means_obs"])[final_range]
    final_mi = np.array(mi_data[model][ds]["mi_corrected"])[final_range]
    final_mi = np.median(final_mi).item()
    # Get corresponding index in 'main' list
    # final_idx = np.where(np.array(mi_data[model][ds]["means_obs"]) == final_mi)[
    #     0
    # ].item()
    final_idx = np.where(np.array(mi_data[model][ds]["mi_corrected"]) == final_mi)[
        0
    ].item()
    # Get confidence interval for it
    final_low = mi_data[model][ds]["lows_obs"][final_idx]
    final_high = mi_data[model][ds]["highs_obs"][final_idx]

    print(
        f"{model.upper()} + {ds}: {final_mi:.3f} -- CI=({final_low:.3f}, {final_high:.3f})"
    )